# Jump-Point 5-Model Classification (32 / 64 / 128 dim)

**Input**: Pre-trained jump-point embeddings (emb_train/val/test.parquet)  
**Pipeline**:
```
Load embeddings (32 / 64 / 128 dim)
  → Combine train+val for GridSearch (stratified 5-fold CV)
  → Tune & evaluate 5 classifiers:
       Logistic Regression, SVM, Decision Tree, Random Forest, XGBoost
  → Save tuned models → 5models/7daysjumppoint_withtime/{dim}dim/
  → Save results CSV + confusion matrix plots
```

---
## Step 1 — Install & Import

In [ ]:
!pip install -q xgboost scikit-learn

In [ ]:
import os
import gc
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from tqdm.auto import tqdm

print('All imports OK')

---
## Step 2 — Mount Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_AI       = "/content/drive/My Drive/Monmouth_University/2025_RA/Medical_AI"

# Where the jump-point embeddings were saved
BASE_EMB      = os.path.join(BASE_AI, "Best_Model")

# Where tuned classifiers will be saved
MODELS_ROOT   = os.path.join(BASE_AI, "5models", "7daysjumppoint_withtime")

EMBED_DIMS    = [32, 64, 128]

print(f'Embeddings root : {BASE_EMB}')
print(f'Models root     : {MODELS_ROOT}')

# Verify embedding folders exist
print('\nChecking embedding folders...')
for dim in EMBED_DIMS:
    folder = os.path.join(BASE_EMB,
                 f'jumppoint_embedding_classifier_{dim}dim_withtime_jumppoint')
    for split in ['emb_train', 'emb_val', 'emb_test']:
        path = os.path.join(folder, f'{split}.parquet')
        status = '✅' if os.path.exists(path) else '❌ MISSING'
        print(f'  {status}  dim={dim}  {split}')

---
## Step 3 — Define Classifiers & Hyperparameter Grids

In [ ]:
CV_FOLDS = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

CLASSIFIERS = {
    'logistic_regression': {
        'estimator': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'solver': ['lbfgs', 'liblinear'],
            'class_weight': [None, 'balanced'],
        },
        'save_name': 'LR',
    },
    'svm': {
        'estimator': SVC(probability=True, random_state=42),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear'],
            'class_weight': [None, 'balanced'],
        },
        'save_name': 'SVM',
    },
    'decision_tree': {
        'estimator': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [3, 5, 10, None],
            'min_samples_split': [2, 5, 10],
            'class_weight': [None, 'balanced'],
        },
        'save_name': 'DT',
    },
    'random_forest': {
        'estimator': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': [None, 'balanced'],
        },
        'save_name': 'RF',
    },
    'xgboost': {
        'estimator': xgb.XGBClassifier(
            eval_metric='logloss', random_state=42,
            use_label_encoder=False),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.8, 1.0],
        },
        'save_name': 'XGB',
    },
}

print(f'Classifiers defined: {list(CLASSIFIERS.keys())}')

---
## Step 4 — Helper Functions

In [ ]:
def evaluate(estimator, name, X_tr, y_tr, pids_tr,
             X_va, y_va, pids_va,
             X_te, y_te, pids_te,
             out_dir, save_name):
    """Evaluate on train / val / test and save confusion matrix."""
    results = {}
    for split, X, y, pids in [
        ('train', X_tr, y_tr, pids_tr),
        ('val',   X_va, y_va, pids_va),
        ('test',  X_te, y_te, pids_te),
    ]:
        pred  = estimator.predict(X)
        proba = estimator.predict_proba(X)[:, 1]
        acc   = accuracy_score(y, pred)
        try:   auc = roc_auc_score(y, proba)
        except: auc = float('nan')
        f1    = f1_score(y, pred, zero_division=0)
        results[split] = {'acc': acc, 'auc': auc, 'f1': f1}

    # Confusion matrix on test set
    pred_te = estimator.predict(X_te)
    cm = confusion_matrix(y_te, pred_te)
    fig, ax = plt.subplots(figsize=(4, 3))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
    ax.set_title(f'{name}\nTest acc={results["test"]["acc"]:.3f}  AUC={results["test"]["auc"]:.3f}')
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'cm_{save_name.lower()}.png'), dpi=150)
    plt.show()
    plt.close()

    return results


def print_results(name, results):
    print(f'  {name}')
    for split in ['train', 'val', 'test']:
        r = results[split]
        print(f'    {split:>6}: acc={r["acc"]:.4f}  auc={r["auc"]:.4f}  f1={r["f1"]:.4f}')


print('Helper functions defined.')

---
## Step 5 — Main Loop: Load Embeddings → Tune → Evaluate → Save

In [ ]:
all_summary = []   # collect one row per (dim, model, split)

for EMBED_DIM in EMBED_DIMS:
    print('\n' + '█'*70)
    print(f'  EMBED_DIM = {EMBED_DIM}')
    print('█'*70)

    # ── Load embeddings ───────────────────────────────────────────────────────
    EMB_FOLDER = os.path.join(
        BASE_EMB,
        f'jumppoint_embedding_classifier_{EMBED_DIM}dim_withtime_jumppoint'
    )
    emb_train = pd.read_parquet(os.path.join(EMB_FOLDER, 'emb_train.parquet'))
    emb_val   = pd.read_parquet(os.path.join(EMB_FOLDER, 'emb_val.parquet'))
    emb_test  = pd.read_parquet(os.path.join(EMB_FOLDER, 'emb_test.parquet'))

    EMB_COLS = [c for c in emb_train.columns if c.startswith('emb_')]
    print(f'  Loaded: train={emb_train.shape}  val={emb_val.shape}  test={emb_test.shape}')
    print(f'  Embedding dims: {len(EMB_COLS)}')

    # ── Build X / y arrays ───────────────────────────────────────────────────
    X_tr   = emb_train[EMB_COLS].values
    y_tr   = emb_train['label'].values
    pids_tr = emb_train['patient_id'].values

    X_va   = emb_val[EMB_COLS].values
    y_va   = emb_val['label'].values
    pids_va = emb_val['patient_id'].values

    X_te   = emb_test[EMB_COLS].values
    y_te   = emb_test['label'].values
    pids_te = emb_test['patient_id'].values

    # Combine train+val for GridSearchCV
    X_tv   = np.vstack([X_tr, X_va])
    y_tv   = np.concatenate([y_tr, y_va])

    # StandardScaler — fit on train only, apply to all
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    X_va   = scaler.transform(X_va)
    X_te   = scaler.transform(X_te)
    X_tv   = scaler.transform(X_tv)   # scaler already fit on X_tr

    # ── Output directories ────────────────────────────────────────────────────
    DIM_DIR = os.path.join(MODELS_ROOT, f'{EMBED_DIM}dim')
    RES_DIR = os.path.join(DIM_DIR, 'results')
    os.makedirs(RES_DIR, exist_ok=True)
    print(f'  Save dir: {DIM_DIR}')

    # ── Run each classifier ───────────────────────────────────────────────────
    dim_records = []

    for clf_name, cfg in CLASSIFIERS.items():
        print(f'\n  [{clf_name}]  GridSearchCV (5-fold, scoring=f1)...')
        t0 = time.time()

        gs = GridSearchCV(
            cfg['estimator'], cfg['params'],
            scoring='f1', cv=CV_FOLDS,
            n_jobs=-1, refit=True, verbose=0
        )
        gs.fit(X_tv, y_tv)
        best_est = gs.best_estimator_
        elapsed  = time.time() - t0

        print(f'  Best params : {gs.best_params_}')
        print(f'  CV F1       : {gs.best_score_:.4f}  ({elapsed:.1f}s)')

        # Evaluate
        results = evaluate(
            best_est, clf_name, X_tr, y_tr, pids_tr,
            X_va, y_va, pids_va, X_te, y_te, pids_te,
            RES_DIR, cfg['save_name']
        )
        print_results(clf_name, results)

        # Save tuned model
        model_path = os.path.join(DIM_DIR, f"{cfg['save_name']}_best.pkl")
        joblib.dump(best_est, model_path)
        print(f'  ✅ Saved → {model_path}')

        # Save scaler once per dim (alongside models)
        scaler_path = os.path.join(DIM_DIR, 'scaler.pkl')
        joblib.dump(scaler, scaler_path)

        # Collect summary rows
        for split in ['train', 'val', 'test']:
            dim_records.append({
                'embed_dim': EMBED_DIM,
                'model': clf_name,
                'split': split,
                'accuracy': results[split]['acc'],
                'auc':      results[split]['auc'],
                'f1':       results[split]['f1'],
                'best_params': str(gs.best_params_),
                'cv_f1':    gs.best_score_,
            })

    # Save per-dim results CSV
    df_dim = pd.DataFrame(dim_records)
    df_dim.to_csv(os.path.join(RES_DIR, 'results.csv'), index=False)
    print(f'\n  Results CSV saved → {RES_DIR}/results.csv')

    all_summary.extend(dim_records)
    del emb_train, emb_val, emb_test, X_tr, X_va, X_te, X_tv
    gc.collect()


# Save combined summary across all dims
df_all = pd.DataFrame(all_summary)
summary_path = os.path.join(MODELS_ROOT, 'all_dims_summary.csv')
df_all.to_csv(summary_path, index=False)
print(f'\n✅ Full summary saved → {summary_path}')

---
## Step 6 — Print Final Summary Table

In [ ]:
df_all = pd.read_csv(os.path.join(MODELS_ROOT, 'all_dims_summary.csv'))

# Test set results only
df_test_summary = (
    df_all[df_all['split'] == 'test']
    [['embed_dim', 'model', 'accuracy', 'auc', 'f1']]
    .sort_values(['embed_dim', 'accuracy'], ascending=[True, False])
    .reset_index(drop=True)
)

print('\n=== TEST SET RESULTS ===')
print(df_test_summary.to_string(index=False))

# Best per dim
print('\n=== BEST MODEL PER DIM (by test accuracy) ===')
best = df_test_summary.loc[
    df_test_summary.groupby('embed_dim')['accuracy'].idxmax()
]
print(best.to_string(index=False))

---
## Step 7 — Verify Saved Models

In [ ]:
print('Verifying saved model files...')
for dim in EMBED_DIMS:
    DIM_DIR = os.path.join(MODELS_ROOT, f'{dim}dim')
    print(f'\ndim={dim}  →  {DIM_DIR}')
    for clf_name, cfg in CLASSIFIERS.items():
        path = os.path.join(DIM_DIR, f"{cfg['save_name']}_best.pkl")
        status = '✅' if os.path.exists(path) else '❌ MISSING'
        print(f'  {status}  {cfg["save_name"]}_best.pkl')
    scaler_path = os.path.join(DIM_DIR, 'scaler.pkl')
    status = '✅' if os.path.exists(scaler_path) else '❌ MISSING'
    print(f'  {status}  scaler.pkl')